# backtrader??

**by Zoe**

<br />
<br />
<br />

backtrader???Python???????,???talib????analyzer??????,???????????????????????????????,?????????????????,??????????????????,???backtrader??????????????
<br />

In [ ]:
from datetime import datetime
import backtrader as bt
import tushare as ts
import pandas as pd
from backtrader.feeds import PandasData

## 1. ????
### 1.1. data collecting

backtrader??Yahoo?online?????????????????????,?wind,tushare??
backtrader??????7???,??datetime,open,high,low,close,volume,openinterest????datetime???index?

????????,???????????,????????????????????,vice versa???????????????????,???????????????????,???????????,https://xueqiu.com/3488649239/62074848 ????????????,????????????:??????????,??????????????

In [ ]:
# tushare
def get_single_kdata(code, start='2020-01-01', end='2021-08-18', index=False):
    df = ts.get_k_data(code, autype='qfq', start=start, end=end, index=index)
    df['date'] = pd.to_datetime(df['date'])
    df['openinterest'] = 0
    return df[['date', 'open', 'high', 'low', 'close', 'volume', 'openinterest']]

# # yahoo
# def get_single_kdata(code, start='2020-01-01', end='2021-08-18'):
#     df = bt.feeds.YahooFinanceData(
#             dataname='AAPL',
#             fromdate=datetime.strptime(start,'%Y-%m-%d'),
#             todate=datetime.strptime(end,'%Y-%m-%d'),
#             adjclose=True,
#             adjvolume=True)
#     return df

In [ ]:
secu_lst = ['600000','000001']
kdata = {}
for secu in secu_lst:
    kdata[secu] = get_single_kdata(secu)
    
print(kdata['000001'].head())

??????,???????????,?????????????

In [ ]:
# ?????????start?end
# ????
secu_lst = {'600000':{'start':'2020-01-01','end':'2020-07-18'},
            '000001':{'start':'2020-02-01','end':'2020-08-18'}}
kdata = {}
for secu in secu_lst.keys():
    kdata[secu] = get_single_kdata(secu, secu_lst[secu]['start'], secu_lst[secu]['end']).reset_index(drop=True)
    
print(kdata['000001'].head())

### 1.2. data cleansing and processing

????????????,??????????????????????????????????backtrader?????next()??all data???????,??????prenext()?????????????,????????????????,?????????

In [ ]:
# ????trading_dates
benchmark_start = min(secu_lst.values(), key=lambda x : x['start'])['start']
benchmark_end = max(secu_lst.values(), key=lambda x : x['end'])['end']
print(f"start: {benchmark_start}, end {benchmark_end}")

benchmark = '399905' # ??500
kdata['benchmark'] = get_single_kdata(benchmark, benchmark_start, benchmark_end, index=True) 
trading_dates = kdata['benchmark']['date'].tolist()
print(f"start: {trading_dates[0]}, end {trading_dates[-1]}")

In [ ]:
for secu in set(kdata.keys())-set(['benchmark']):
    print(secu)
    secu_kdata = kdata['benchmark'][['date']].merge(kdata[secu],how='left')
    secu_kdata['suspend'] = 0 
    secu_kdata.loc[secu_kdata['open'].isnull(), 'suspend'] = 1 # ??????
    secu_kdata.set_index(['date'], inplace = True) # ?date?index
    end = secu_lst[secu]['end']
    secu_kdata.fillna(method='ffill',inplace = True) # start?????????????
    secu_kdata.fillna(value = 0,inplace = True) #start?????0??
    secu_kdata.loc[(secu_kdata.index > end), 'suspend'] = 1
    print(secu_kdata)
    kdata[secu] = secu_kdata

?????????,?????ok,????,end????????bar????
?????class??????

In [ ]:
class GetKdatas(object):
    def __init__(self, secu_lst, benchmark='000001'):
        """
        :parameter secu_lst: a dict contained stocks with starts and ends
        :parameter benchmark: the name of benchmark
        """
        self.secu_lst = secu_lst
        self.benchmark = benchmark

    @staticmethod
    def get_single_kdata(code, start='2020-01-01', end='2021-08-18', index=False):
        df = ts.get_k_data(code, autype='qfq', start=start, end=end, index=index)
        df['date'] = pd.to_datetime(df['date'])
        df['openinterest'] = 0
        return df[['date', 'open', 'high', 'low', 'close', 'volume', 'openinterest']]

    def get_all_kdata(self):
        kdata = {}
        for secu in set(self.secu_lst):
            secu_kdata = self.get_single_kdata(secu, self.secu_lst[secu]['start'], self.secu_lst[secu]['end'])
            kdata[secu] = secu_kdata.reset_index(drop=True)
        return kdata

    def merge_period(self):
        all_kdata = self.get_all_kdata()
        benchmark_start = min(self.secu_lst.values(), key=lambda x: x['start'])['start']
        benchmark_end = max(self.secu_lst.values(), key=lambda x: x['end'])['end']
        all_kdata['benchmark'] = self.get_single_kdata(self.benchmark, benchmark_start, benchmark_end,True)

        for secu in set(all_kdata.keys()) - set(['benchmark']):
            secu_kdata = all_kdata['benchmark'][['date']].merge(all_kdata[secu], how='left')
            secu_kdata['suspend'] = 0
            secu_kdata.loc[secu_kdata['open'].isnull(), 'suspend'] = 1  # ??????
            secu_kdata.set_index(['date'], inplace=True)  # ?date?index
            end = secu_lst[secu]['end']
            secu_kdata.fillna(method='ffill', inplace=True)  # start?????????????
            secu_kdata.fillna(value=0, inplace=True)  # start?????0??
            secu_kdata.loc[(secu_kdata.index > end), 'suspend'] = 1
            all_kdata[secu] = secu_kdata

        _ = all_kdata.pop('benchmark')
        return all_kdata

In [ ]:
secu_lst = {'600000': {'start': '2020-01-01', 'end': '2020-07-18'},
            '000001': {'start': '2020-02-01', 'end': '2020-08-18'}}

kdata = GetKdatas(secu_lst).merge_period()
print(kdata)


### 1.3. data feeding

?????????,backtrader???datetime,open,high,low,close,volume,openinterest?7??
<br/>
?????open (default: 1) , high (default: 2), low (default: 3), close (default: 4), volume (default: 5), openinterest (default: 6)?
<br/>
??,??????add???*name=secu*,????????????

In [ ]:
cerebro = bt.Cerebro()
for secu in set(kdata.keys())-set(['benchmark']):
    df = PandasData(dataname=kdata[secu], fromdate=kdata[secu].index[0],todate=kdata[secu].index[-1])
    cerebro.adddata(df, name=secu) 

# ?????csv,????column???????,??datetime??????,??????
# data = btfeeds.GenericCSVData(
#     dataname='mydata.csv',
#     fromdate=benchmark_start,
#     todate=benchmark_end,
#     dtformat=('%Y-%m-%d'),
#     datetime=0,
#     high=1,
#     low=2,
#     open=3,
#     close=4,
#     volume=5,
#     openinterest=-1
# )

**???????????,???????????????????PandasData_Extend????load????,??PandasData??GenericCSVData,?????8???

In [ ]:
class PandasData_Extend(PandasData):
     # Add a 'suspend' line to the inherited ones from the base class
    lines = ('suspend',)
    # ???(open,0), ... , (openinterest,5)?6?,????1
    # add the parameter to the parameters inherited from the base class
    params = (('suspend', 6),)
    
    # ???csv??,openinterest ? GenericCSVData index?7,??1
    # params = (('suspend', 8),)


## 2. ????
### 2.1. ???????????

??????10000,?????,?????????????????????,??????????????,???????????????,??????????????,???????????

In [ ]:
cerebro = bt.Cerebro()
# ???????1 million
startcash = 10**6
cerebro.broker.setcash(startcash)
print(f"????{cerebro.broker.getvalue()}")

????????,?????????,???????????????????,??????????,?????????????????
<br/>
?????,????????????????,???????

In [ ]:
class CommInfoPro(bt.CommInfoBase):

    params = (
        ('stamp_duty', 0.001),  # ????
        ('stamp_duty_fe', 1),  # ?????
        ('commission', 0.001),  # ???
        ('commission_fee', 5),  # ?????
        ('stocklike', True), # ??
        ('commtype', bt.CommInfoBase.COMM_PERC), # ????
    )

    def _getcommission(self, size, price, pseudoexec):
        '''
        If size is greater than 0, this indicates a long / buying of shares.
        If size is less than 0, it idicates a short / selling of shares.
        '''

        if size > 0:  # ??,??????
            return max(size * price * self.p.commission, self.p.commission_fee)
        elif size < 0:  # ??,?????
            return max(size * price * (self.p.stamp_duty + self.p.commission), self.p.stamp_duty_fe)
        else:
            return 0  # just in case for some reason the size is 0.

        
cerebro.broker.addcommissioninfo(CommInfoPro())

### 2.2. ??????
????,A?????100????,??order size???100??????????????????????????????backtrader????order??,???amount???downcast,??jupyter??????????????order???????amount????

??:????,??????

In [ ]:
# ??
#self.buy(data, amount)
#self.sell(data, amount)

#??
def downcast(amount, lot):
    return abs(amount//lot*lot)

print(downcast(418,100))
print(downcast(-418,100))
#self.buy(data, downcast(amount, lot))
#self.sell(data, downcast(amount, lot))

### 2.3. ??????

backtrader???????data._name,????????????????????linseries.py??????@property
<img src="img1.png" width="40%">

In [ ]:
# in linseries.py
# @property
# def name(self):
#     return self._name

### 2.4 data?line???

line?backtrader???,?????????????line??lines,???data feed?????open(???)?close(???)?6?line????line????????,0???????,-1?????bar???,-2?????bar???;??,1?????bar???,2?????bar???,?????

data?line???,?line??????data??????(??????)???,??open?close?lines,???data??list???datas????????data?,????self.datas[x]??self.datax,x????????????(?0??),?????self.datasbyname[data._name]???

backtrader???????????????data0?close??,??:self.data0.close[0],self.data0_close[0],self.data0_0[0](???0??close????0)?

## 3. ????
### 3.1. ????
????????????????????????>20??????,?????
??????,???init???,????????????next???????
1. self.params.your_param???self.params.maperiod
2. self.datas?list??,???????,?????1?self.datas[0]???????????


?backtrader?order???,????????name????????order?class,????,?name???????
<br/>
??????,??????????
1. self.order_target_percent(secu_data, target_pct, name=secu)
2. self.order_target_value(secu_data, target_val, name=secu)
3. self.buy(secu_data, order_amount, name=secu)
4. self.sell(secu_data, order_amount, name=secu)

??:???????????,?????????????backtrader????????????/?????,??????????????????????????????????????,???????? ???????????????
<br/>
??:???????bar?????,??????bar,???end_date????????????????bar??????????,???????????????,??????????(end_date)?????

In [ ]:
class single_strategy(bt.Strategy):
    # ???????????
    params = (
        ('maperiod', 20),
    )

    def __init__(self):
        # ???????
        self.order = None

        # ????????,???talib??
        self.sma = bt.ind.SMA(self.datas[0], period=self.params.maperiod)

    # ????,?????????,???????
    def prenext(self):
        pass
    
    def downcast(amount, lot):
        return abs(amount//lot*lot)
    
    
    def next(self):
        if self.order:  # ???????????,?????????bar
            return
        
        # ???????????
        if self.datas[0].datetime.date(0) == end_date:
            return 
        
        # ???bar?????????(????K????,??????????)
        self.log("%.2f?" % self.broker.getvalue()) 
        if not self.position:  # ????
            
            # ????????:????????20???;
            # ???????????????
            if self.datas[0].close > self.sma and data.datetime.date(1) < end_date:
                # ????????????
                order_value = self.broker.getvalue()*0.98
                order_amount = downcast(order_value/self.datas[0].close[0], 100)
                self.order = self.buy(self.datas[0], size=order_amount, name=self.datas[0]._name)
                self.log(f"?{self.datas[0]._name}, price:{self.datas[0].close[0]:.2f}, amout: {order_amount}")
                # self.order = self.order_target_percent(self.datas[0], 0.98, name=self.datas[0]._name)
                # self.log(f"?{self.datas[0]._name}, price:{self.datas[0].close[0]:.2f}, pct: 0.98")
        else:
            
            # ????????:??????20???,??????
            if self.datas[0].close > self.sma or data.datetime.date(1) >= end_date:
                # ????
                self.order = self.order_target_percent(self.datas[0], 0, name=self.datas[0]._name)
                self.log(f"?{self.datas[0]._name}, price:{self.datas[0].close[0]:.2f}, pct: 0")

    def log(self, txt, dt=None):
        ''' ????'''
        dt = dt or self.datas[0].datetime.date(0) # ??????
        print('%s, %s' % (dt.isoformat(), txt))

    def notify_order(self, order):
        if order.status in [order.Submitted, order.Accepted]:
            # Buy/Sell order submitted/accepted to/by broker - Nothing to do
            return

        # Check if an order has been completed
        # Attention: broker could reject order if not enough cash
        if order.status in [order.Completed, order.Canceled, order.Margin]:
            if order.isbuy():
                self.log(f"""??{order.info['name']}, ???{order.executed.size},???{order.executed.price:.2f}""")
            elif order.issell():
                self.log(f"""??{order.info['name']}, ???{order.executed.size},???{order.executed.price:.2f}""")
            self.bar_executed = len(self)

        # Write down: no pending order
        self.order = None


cerebro = bt.Cerebro()
cerebro.addstrategy(single_strategy)
secu_lst = {'600000': {'start': '2020-01-01', 'end': '2020-07-18'}}
df = GetKdatas(secu_lst).merge_period()['600000']
data = PandasData_Extend(dataname=df, fromdate=df.index[0], todate=df.index[-1])
cerebro.adddata(data, name='600000')
end_date = df.index[-1] # ?????

# ???????1 million
startcash = 10**6
cerebro.broker.setcash(startcash)
print(f"????{cerebro.broker.getvalue()}")
# ???????
cerebro.broker.addcommissioninfo(CommInfoPro())
# ??????
cerebro.run()
# ???????????
portvalue = cerebro.broker.getvalue()
# ????
print(f'????: {round(portvalue, 2)}')

### 3.2. ????
???????????
?????????dict????????????
1. self.getdatanames()????????????list
2. self.getdatabyname(secu_name):??????data

In [ ]:
class multi_strategy(bt.Strategy):
    # ???????????
    params = (
        ('maperiod', 20),
    )

    def __init__(self):
        # ???????
        self.order = None
        self.buy_lst = []

        # ????????,???talib??
        # ???????????
        self.sma = {x: bt.ind.SMA(self.getdatabyname(x), period=self.p.maperiod) for x in self.getdatanames()}

    def prenext(self):
        pass
    
        
    def downcast(self, amount, lot):
        return abs(amount//lot*lot)
    
    
    def next(self):
        if self.order:  # ???????????,
            return

        # ???????????
        if self.datas[0].datetime.date(0) == end_date:
            return 
        
        # ??????
        self.log(f'{self.broker.getvalue():.2f}, {[(x, self.getpositionbyname(x).size) for x in self.buy_lst]}')
        if len(self.buy_lst) < 2:  # ????
            for secu in set(self.getdatanames()) - set(self.buy_lst):
                data = self.getdatabyname(secu)
                # ????????:????????20???
                # ???????????????
                if data.close > self.sma[secu] and \
                    data.datetime.date(1) < pd.Timestamp(secu_lst[secu]['end']):
                    # ????
                    order_value = self.broker.getvalue()*0.48
                    order_amount = self.downcast(order_value/data.close[0], 100)
                    self.order = self.buy(data, size=order_amount, name=secu)
                    self.log(f"?{secu}, price:{data.close[0]:.2f}, amout: {order_amount}")
                    self.buy_lst.append(secu)
        elif self.position:
            now_lst = []
            for secu in self.buy_lst:
                data = self.getdatabyname(secu)
                # ????????:??????20???,??????
                if (data.close < self.sma[secu]) or \
                    (data.datetime.date(1) >= pd.Timestamp(secu_lst[secu]['end'])):
                    # ????
                    self.order = self.order_target_percent(data, 0, name=secu)
                    # ???? self.sell(data, size = self.getposition(data).size)
                    # or self.sell(data, size = self.getpositionbyname(secu).size)
                    self.log(f"?{secu}, price:{data.close[0]:.2f}, pct: 0")
                    continue
                now_lst.append(secu)
            self.buy_lst = now_lst.copy()

    def log(self, txt, dt=None):
        ''' ????'''
        dt = dt or self.datas[0].datetime.date(0)
        print('%s, %s' % (dt.isoformat(), txt))

    def notify_order(self, order):
        if order.status in [order.Submitted, order.Accepted]:
            # Buy/Sell order submitted/accepted to/by broker - Nothing to do
            return

        # Check if an order has been completed
        # Attention: broker could reject order if not enough cash
        if order.status in [order.Completed, order.Canceled, order.Margin]:
            if order.isbuy():
                self.log(f"""??{order.info['name']}, ???{order.executed.size},???{order.executed.price:.2f}""")
            elif order.issell():
                self.log(f"""??{order.info['name']}, ???{order.executed.size},???{order.executed.price:.2f}""")
            self.bar_executed = len(self)

        # Write down: no pending order
        self.order = None

        
secu_lst = {'600000': {'start': '2020-01-01', 'end': '2021-07-20'},
            '000001': {'start': '2020-02-01', 'end': '2021-08-18'}}
# ??????
kdata = GetKdatas(secu_lst).merge_period()
kdata = dict(sorted(kdata.items()))

# ????
cerebro = bt.Cerebro()
cerebro.addstrategy(multi_strategy)

for secu in kdata.keys():
    df = kdata[secu]
    data = PandasData_Extend(dataname=df, fromdate=df.index[0], todate=df.index[-1])
    cerebro.adddata(data, name=secu)
end_date = df.index[-1]

# ???????1 million
startcash = 10**6
cerebro.broker.setcash(startcash)
print(f"????{cerebro.broker.getvalue()}")
# ???????
cerebro.broker.addcommissioninfo(CommInfoPro())

# ????
cerebro.addanalyzer(bt.analyzers.SharpeRatio, _name='_sharpe')
cerebro.addanalyzer(bt.analyzers.AnnualReturn, _name='_annrtn')
cerebro.addanalyzer(bt.analyzers.DrawDown, _name='_dd')
cerebro.addanalyzer(bt.analyzers.PyFolio, _name='_pyfolio')
# ??????
thestrats = cerebro.run()
# ???????????
portvalue = cerebro.broker.getvalue()
# ????
print(f'????: {round(portvalue, 2)}')

## 4. ????
???????????????????,????????????????

In [ ]:
cerebro.plot(style='candlestick')

??????????????????????????????,???252???????????????,????????,??252???12?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

thestrat = thestrats[0]
pyfolio = thestrat.analyzers._pyfolio.get_analysis()
def plot_strategy(pyfolio):
    returns = pyfolio['returns'].values()
    returns = pd.DataFrame(list(zip(pyfolio['returns'].keys(),pyfolio['returns'].values())), columns=['date','total_value'])
    

    sharpe = np.round(np.sqrt(252) * returns['total_value'].mean() / returns['total_value'].std(), 4)
    returns['total_value']=returns['total_value']+1
    returns['total_value'] = returns['total_value'].cumprod()
    annal_rtn = np.round(returns['total_value'].iloc[-1]**(252/len(returns))-1, 4)*100
    dd = 1-returns['total_value']/np.maximum.accumulate(returns['total_value'])
    end_idx = np.argmax(dd)
    start_idx = np.argmax(returns['total_value'].iloc[:end_idx])
    maxdd_days = end_idx-start_idx
    maxdd = np.round(max(dd), 4)*100
    
    returns = returns.set_index('date')
    ax=returns.plot(y='total_value')
    plt.text(0.01,0.8, f'sharpe: {sharpe:.2f}', transform=ax.transAxes)
    plt.text(0.01,0.75, f'maxdd: {maxdd:.2f}%', transform=ax.transAxes)
    plt.text(0.01,0.7, f'maxdd_days: {maxdd_days:}days', transform=ax.transAxes)
    plt.text(0.01,0.65, f'annal_rtn: {(annal_rtn):.2f}%', transform=ax.transAxes)
    plt.scatter([returns.index[start_idx], returns.index[end_idx]], [returns.iloc[start_idx], returns.iloc[end_idx]],
                s = 80, c = 'g', marker = 'v', label = 'MaxDrawdown Duration')
    plt.title('portfolio')
    plt.show()

plot_strategy(pyfolio)